# Weekly Project 06 - 3D Point Cloud Processing 2

Today we will continue the work from Monday.
We will follow the style of last week.

Weekly project:

- You will need to implement your own k-means algorithm. You are not allowed to use the implementation in `sklearn`.
- It should be able to cluster each of the different figures.
- Extend your k-means algorithm so that it finds the optimal number of clusters.

## Challenge

- Implement the mean shift clustering algorithm.


## Weekly Project 06 - Start Here

In [1]:
import numpy as np
import open3d as o3d
import copy
import matplotlib.pyplot as plt

def draw_labels_on_model(pcl, labels):
    cmap = plt.get_cmap("tab20")
    pcl_temp = copy.deepcopy(pcl)
    max_label = labels.max()
    colors = cmap(labels / (max_label if max_label > 0 else 1))
    colors[labels < 0] = 0
    pcl_temp.colors = o3d.utility.Vector3dVector(colors[:, :3])
    o3d.visualization.draw_geometries([pcl_temp])

d = 4
mesh = o3d.geometry.TriangleMesh.create_tetrahedron().translate((-d, 0, 0))
mesh += o3d.geometry.TriangleMesh.create_octahedron().translate((0, 0, 0))
mesh += o3d.geometry.TriangleMesh.create_icosahedron().translate((d, 0, 0))
mesh += o3d.geometry.TriangleMesh.create_torus().translate((-d, -d, 0))
mesh += o3d.geometry.TriangleMesh.create_mobius(twists=1).translate((0, -d, 0))
mesh += o3d.geometry.TriangleMesh.create_mobius(twists=2).translate((d, -d, 0))

# Apply k-means to this point cloud
point_cloud = mesh.sample_points_uniformly(int(1e3))

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
def kmeans(points, k, n_iter=200):
    # Randomly initialize k cluster centers uniformly inside the bounding box of the point cloud
    centers = np.random.uniform(points.min(axis=0), points.max(axis=0), size=(k, points.shape[1]))
    for _ in range(n_iter):
        # Assign each point to its nearest center
        dists = np.linalg.norm(points[:, None, :] - centers[None, :, :], axis=2)  # (N, k)
        labels = np.argmin(dists, axis=1)
        # Move each center to the mean of its points (keep the old center if the cluster is empty)
        new_centers = np.array([points[labels == q].mean(axis=0) if np.any(labels == q) else centers[q]
                                for q in range(k)])
        if np.allclose(new_centers, centers):
            break
        centers = new_centers
    return labels

In [3]:
def draw_labels_on_model(pcl,labels):
    cmap = plt.get_cmap("tab20")
    pcl_temp = copy.deepcopy(pcl)
    max_label = labels.max()
    print("%s has %d clusters" % (pcl_name, max_label + 1))
    colors = cmap(labels / (max_label if max_label > 0 else 1))
    colors[labels < 0] = 0
    pcl_temp.colors = o3d.utility.Vector3dVector(colors[:, :3])
    o3d.visualization.draw_geometries([pcl_temp])

In [4]:
pcl_name = 'pcl'
#km = KMeans(n_clusters=6, init='random',
#            n_init=10, max_iter=300, tol=1e-04, random_state=0)

# Get the points from the pointcloud as nparray
xyz = np.asarray(point_cloud.points)
k = 6
labels = kmeans(xyz, k)
draw_labels_on_model(point_cloud, labels)

pcl has 6 clusters


In [ ]:
k_values = range(1, 11)
distortions = []
for k in k_values:
    labels, centers, distortion = kmeans(
        xyz,
        k=k,
        random_state=0
    )
    distortions.append(distortion)   
plt.plot(k_values, distortions, marker = "o")
plt.xticks(list(k_values))
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Distortion")
plt.title("K-Means Clustering")
plt.show()